# CS781 Capstone - Phase 1: Arch 2 Candidate Pool x48 -> 18 Submissions (inference only)

- For every prompt, one batched pass generates 48 candidates from Arch 2 (all `t35`:
- T 0.5, top-p 0.95, top-k 20, rp 1.2).
- Eighteen MBR selectors (ROUGE-1/2 consensus) pick from pools of 12, 16, 20, 24, 32, 40 and 48 candidates, with independent repeats at each size.
- All 18 are scored on the 200 held-out tuning prompts, then written as validated zips `NN_submission_<selector>.zip` in held-out rank order.
- `USE_CONTINUED = True` (config cell) uses the continued Arch 2 instead.

Attach: the tokenizer dataset and `arch2-trained` (and the continuation output if used). GPU T4, Internet on, `HF` secret.

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import hashlib
import glob
import shutil
import sys
import numpy as np
from collections import Counter

NOTEBOOK_START = time.time()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- which trained model family to select checkpoints / decoding for ----
MODEL_KIND = "arch2"            # "arch2" (original + continued + snapshots) or "arch4"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
VOCAB_SIZE = 16384
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
MODEL_SEQ_LEN = 1024

# Held-out evaluation set (fixed for every checkpoint and every decoding trial)
#  arch2: rebuilt by the same exact replay as the continuation notebook (documents neither Arch 2 nor the continued run
#         trains on); the tuning half is identical to the continuation notebook's 200 held-out prompts.
#  arch4: Arch 4's own eval set (validation documents only), loaded from its output.
#  Selection (checkpoints, decoding, MBR) uses only the tuning set; the audit set is scored once at the end.
EVAL_N = 400                      # 50 per source for tuning + the next 50 per source for a final audit
A2_TRAIN_TOKENS, A2_VAL_TOKENS, HELDOUT_PCT = 300_000_000, 5_000_000, 5   # must match the continuation notebook

# Generation regime (same for every trial so comparisons are fair)
TARGET_WORDS = 128
PROMPT_KEEP = 704                 # most recent prompt tokens kept; prompt + MAX_NEW_TOKENS fits 1024 without sliding
MAX_NEW_TOKENS = MODEL_SEQ_LEN - PROMPT_KEEP   # 320, covers 128 words at the 99th percentile (~202 tokens)

# Checkpoint selection: every candidate is scored with this reference decoding (Arch 2's submitted settings)
REFERENCE_DECODING = {"temperature": 0.4, "top_p": 0.92, "top_k": 50, "repetition_penalty": 1.15,
                      "rep_mode": "arch2_original", "no_repeat_ngram": 0}

USE_CONTINUED = False             # True: use arch2_continued instead of the original Arch 2 (when attached)

OUTPUT_DIR = "/kaggle/working/pool48_arch2"
INPUT_ROOT = "/kaggle/input"
os.makedirs(OUTPUT_DIR, exist_ok=True)

_tok_matches = glob.glob("/kaggle/input/**/legal_bpe_16384.json", recursive=True)
TOKENIZER_PATH = _tok_matches[0] if _tok_matches else "/kaggle/input/legal-bpe-tokenizer/legal_bpe_16384.json"

def find_input(name):
    m = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    return m[-1] if m else None

def save_json(obj, name):
    tmp = os.path.join(OUTPUT_DIR, name + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=float)
    os.replace(tmp, os.path.join(OUTPUT_DIR, name))

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = torch.cuda.is_available()
def amp_ctx():
    return torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp)
print(f"MODEL_KIND={MODEL_KIND} | device={device} | tokenizer {TOKENIZER_PATH} | outputs -> {OUTPUT_DIR}")


## 4. BPE Tokenizer (from A1)

In [ ]:
%%writefile bpe_tokenizer.py
"""
BPE Tokenizer - CS781 Assignment 1 implementation, optimized for Phase 1.

Speedups over original A1 code (same algorithm):
  1. fit() uses word frequencies + incremental pair-count updates
  2. encode() uses priority-based merging + word-level cache
  3. Checkpointing: saves progress every N merges so training survives Kaggle timeouts

Research-backed defaults:
  - vocab_size=16384: optimal for 100M-scale models (Zheng et al. 2024,
    "Scaling Laws with Vocabulary"). At 100M scale, 16K minimizes training
    cost; quality spread across 8K-256K is <2% BPB.
  - Corpus sample: 5-10M words is sufficient for merge stabilization.
"""

import json
import os
import time
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # ------------------------------------------------------------------ #
    #  Training (with checkpointing)                                      #
    # ------------------------------------------------------------------ #

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair

            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # ------------------------------------------------------------------ #
    #  Encoding                                                           #
    # ------------------------------------------------------------------ #

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}

        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)

        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i

            if best_idx == -1:
                break

            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]

        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()

        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # ------------------------------------------------------------------ #
    #  Decoding                                                           #
    # ------------------------------------------------------------------ #

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()

        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")

        return b"".join(parts).decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    #  Save / Load                                                        #
    # ------------------------------------------------------------------ #

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size

In [ ]:
from bpe_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(TOKENIZER_PATH)
assert tokenizer.vocab_size == VOCAB_SIZE
print(f'Loaded tokenizer: vocab_size={tokenizer.vocab_size}')
os.makedirs('arch2_src', exist_ok=True)
os.makedirs('arch4_src', exist_ok=True)


## 5. Model Code: Arch 2 (unchanged from the Arch 2 notebook)

In [ ]:
%%writefile arch2_src/attention.py
"""Grouped-Query Attention with Rotary Position Embeddings (RoPE)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    half = d_head // 2
    freqs = 1.0 / (base ** (torch.arange(0, half).float() / half))
    t = torch.arange(max_seq_len).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    T = x.shape[2]
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class CausalSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        cos, sin = build_rope_cache(max_seq_len, self.d_head)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out

In [ ]:
%%writefile arch2_src/model_arch.py
"""LLaMA-style Decoder LM: RMSNorm, SwiGLU FFN, RoPE (in attention), GQA."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 12
    n_heads: int = 10
    n_kv_heads: int = 2
    d_ff: int = 1706
    max_seq_len: int = 1024
    dropout: float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads,
                                        cfg.max_seq_len, cfg.dropout)
        self.ln2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

## Held-out prompts (same as the decoding notebook)

In [ ]:
try:
    import ijson
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ijson', '-q'], check=True)
    import ijson
import array
import gc
from huggingface_hub import hf_hub_download

BLOCK = MODEL_SEQ_LEN + 1

if MODEL_KIND == "arch4":
    # current Arch 4 notebook writes arch4v2_* (versioned cache); older runs wrote arch4_*
    p_eval = find_input("arch4v2_eval_set.json") or find_input("arch4_eval_set.json")
    p_val = find_input("arch4v2_val_blocks.npy") or find_input("arch4_val_blocks.npy")
    assert p_eval and p_val, "attach the Arch 4 training output (arch4v2_eval_set.json, arch4v2_val_blocks.npy)"
    with open(p_eval) as f:
        eval_set = json.load(f)
    val_blocks = np.load(p_val, mmap_mode="r")
    EVAL_SOURCE = f"Arch 4 validation-document eval set ({p_eval})"
else:
    if True:
        # Same exact replay as the continuation notebook (same seeds and draws), but without collecting new
        # training tokens: it only needs Arch 2's validation tokens and the held-out prompts after each stop point.
        print("Rebuilding the held-out set by replaying Arch 2's data pipeline...")
        a2_rng = random.Random(SEED)
        val_tok = array.array('H')
        eval_set, stats, earlier_keys = [], {}, set()
        a2_train_ps = A2_TRAIN_TOKENS // len(TRAIN_FILES)
        a2_val_ps = A2_VAL_TOKENS // len(TRAIN_FILES)
        eval_ps = EVAL_N // len(TRAIN_FILES)
        min_words = TARGET_WORDS + 512
        for si, train_file in enumerate(TRAIN_FILES):
            src = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
            s = stats[src] = Counter()
            file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=train_file, repo_type="dataset", token=HF_TOKEN)
            b_rng = random.Random(SEED * 1000 + si)
            seen, a2_train_len, phase = set(), 0, "A"
            val_limit = len(val_tok) + a2_val_ps
            eval_start = len(eval_set)
            with open(file_path, 'rb') as f:
                for record in ijson.items(f, 'item'):
                    s["docs_scanned"] += 1
                    text = record["text"]
                    h = hashlib.md5(text[:500].encode()).hexdigest()
                    if h in seen:
                        continue
                    seen.add(h)
                    if len(text.split()) < 50:
                        continue
                    if phase == "A":
                        toks = tokenizer.encode(text)
                        earlier_keys.add(h)
                        if a2_rng.random() < 0.05:
                            if len(val_tok) < val_limit:
                                val_tok.extend(toks[:val_limit - len(val_tok)])
                        else:
                            if a2_train_len < a2_train_ps:
                                a2_train_len += min(len(toks), a2_train_ps - a2_train_len)
                        if a2_train_len >= a2_train_ps and len(val_tok) >= val_limit:
                            phase = "B"
                            s["A_stop_doc"] = s["docs_scanned"]
                            tokenizer._encode_cache = {}
                        continue
                    if h in earlier_keys:
                        continue
                    if b_rng.random() < HELDOUT_PCT / 100:
                        words = text.split()
                        if len(eval_set) - eval_start < eval_ps and len(words) >= min_words:
                            st = b_rng.randint(0, len(words) - min_words)
                            eval_set.append({"source": src, "doc_key": h, "prompt": " ".join(words[st:st + 512]),
                                             "reference": " ".join(words[st + 512:st + min_words])})
                    if len(eval_set) - eval_start >= eval_ps:
                        break
            print(f"  {src}: Arch 2 stop doc {s.get('A_stop_doc')} | held-out prompts {len(eval_set) - eval_start}")
            tokenizer._encode_cache = {}
            gc.collect()
        n_va = len(val_tok) // BLOCK
        val_blocks = np.frombuffer(val_tok, dtype=np.uint16)[:n_va * BLOCK].reshape(n_va, BLOCK).copy()
        EVAL_SOURCE = "rebuilt by exact replay (identical to the continuation notebook's held-out set)"
        save_json(eval_set, "heldout_eval_set.json")
        np.save(os.path.join(OUTPUT_DIR, "heldout_val_blocks.npy"), val_blocks)

_by_src, _groups = {}, set()
for _e in eval_set:                  # one example per document group, so tuning and audit never share one
    if _e["doc_key"] in _groups:
        continue
    _groups.add(_e["doc_key"])
    _by_src.setdefault(_e["source"], []).append(_e)
_half = min(len(v) for v in _by_src.values()) // 2 if MODEL_KIND == "arch4" else EVAL_N // 8
_half = min(_half, EVAL_N // 8)
TUNE_SET = [e for src in _by_src for e in _by_src[src][:_half]]
AUDIT_SET = [e for src in _by_src for e in _by_src[src][_half:2 * _half]]
assert not ({e["doc_key"] for e in TUNE_SET} & {e["doc_key"] for e in AUDIT_SET}), "tuning and audit share a document group"
eval_set = TUNE_SET                  # every selection step below uses the tuning set only
save_json({"source": EVAL_SOURCE, "tune_doc_keys": [e["doc_key"] for e in TUNE_SET],
           "audit_doc_keys": [e["doc_key"] for e in AUDIT_SET]}, "eval_set_ids.json")
print(f"Held-out prompts ({EVAL_SOURCE}): tuning {len(TUNE_SET)} | audit {len(AUDIT_SET)} | validation blocks {len(val_blocks):,}")


## 8. Candidate Checkpoints

In [ ]:
import unicodedata
import types
import torch.nn.functional as F

sys.path.insert(0, os.path.abspath(f"{MODEL_KIND}_src"))
for _m in ("model_arch", "attention", "rope"):
    sys.modules.pop(_m, None)
from model_arch import LMConfig, DecoderLM

ARCH_NAMES = {
    "arch2": {"llama_style_rope_rmsnorm_swiglu_gqa": "arch2_original",
              "llama_style_rope_rmsnorm_swiglu_gqa_continued": "arch2_continued"},
    "arch4": {"deep_decoder_mla_qknorm_gated_attention_engram": "arch4_final"},
}[MODEL_KIND]
EXTRA_FILES = {"arch2": ["arch2c_snap_*.pt"],
               "arch4": ["arch4_best_phase1_model.pt", "arch4_before_task_segment.pt", "arch4_snap_*.pt"]}[MODEL_KIND]

def discover_candidates():
    """Every trained checkpoint of this family attached as an input: final models (identified by the architecture
    name in the sibling metrics.json) plus snapshots / best / pre-task-segment checkpoints."""
    cands = {}
    for p in sorted(glob.glob(f"{INPUT_ROOT}/**/model.pt", recursive=True)):
        try:
            with open(os.path.join(os.path.dirname(p), "metrics.json")) as f:
                arch = json.load(f).get("architecture")
        except Exception:
            continue
        if arch in ARCH_NAMES:
            cands[ARCH_NAMES[arch]] = p
    for pat in EXTRA_FILES:
        for p in sorted(glob.glob(f"{INPUT_ROOT}/**/{pat}", recursive=True)):
            cands[os.path.basename(p).replace(".pt", "")] = p
    return cands

def engram_compress_map():
    """The exact map the Engram tables were trained with: saved by the training run (arch4_engram_compress.pt, also
    stored inside arch4_latest.pt). Falls back to recomputing the UTF-8-safe map only if neither is attached."""
    p = find_input("arch4_engram_compress.pt")
    if p:
        print(f"Engram compression map: {p}")
        return torch.load(p, map_location="cpu")
    p = find_input("arch4_latest.pt")
    if p:
        print(f"Engram compression map: from {p}")
        return torch.load(p, map_location="cpu", weights_only=False)["engram_compress"].cpu()
    print("WARNING: no saved Engram compression map attached; recomputing (must match the training run)")
    def _key(b, ew):
        try:
            s = b.decode("utf-8")
        except UnicodeDecodeError:
            return ("bytes", b, ew)
        s = unicodedata.normalize("NFKC", s)
        s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
        return (" ".join(s.lower().split()), ew)
    tokenizer.decode([5])
    keys, out = {}, []
    for t in range(VOCAB_SIZE):
        b = tokenizer._token_bytes.get(t)
        key = ("special", t) if b is None else _key(b, bool(tokenizer._ends_word.get(t, False)))
        out.append(keys.setdefault(key, len(keys)))
    return torch.tensor(out, dtype=torch.long)

def _arch2_step(self, idx, cache=None, pad=None):
    """KV-cached forward for Arch 2 (generation only). Uses Arch 2's own modules and weights and repeats the exact
    arithmetic of its attention (RoPE, GQA repeat, scaled scores, causal mask, softmax), so the logits equal a full
    forward over the whole sequence; only the keys/values of earlier tokens are reused. Arch 2's model code is not
    modified. pad (B,) = number of left-padding tokens per row (first call only): each row then gets RoPE positions
    counted from its first real token and never attends to padding, so a padded row computes what it would alone.
    Returns (logits for idx, cache)."""
    pos = 0 if cache is None else cache["pos"]
    B, T = idx.shape
    if cache is not None:
        pad = cache["pad"]
    elif pad is None:
        pad = torch.zeros(B, dtype=torch.long, device=idx.device)
    S = pos + T
    q_abs = torch.arange(pos, S, device=idx.device)
    k_abs = torch.arange(S, device=idx.device)
    positions = (q_abs[None, :] - pad[:, None]).clamp(min=0)                       # (B, T)
    allowed = (k_abs[None, None, :] <= q_abs[None, :, None]) & (                   # causal, and never a pad key
        (k_abs[None, None, :] >= pad[:, None, None]) | (k_abs[None, None, :] == q_abs[None, :, None]))
    x = self.tok_emb(idx)
    new_kv = []
    for li, blk in enumerate(self.blocks):
        a = blk.attn
        h = blk.ln1(x)
        q = a.q_proj(h).view(B, T, a.n_heads, a.d_head).transpose(1, 2)
        k = a.k_proj(h).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
        v = a.v_proj(h).view(B, T, a.n_kv_heads, a.d_head).transpose(1, 2)
        cos = a.rope_cos[positions].unsqueeze(1)                                   # (B, 1, T, d_head/2)
        sin = a.rope_sin[positions].unsqueeze(1)
        d = a.d_head // 2
        q = torch.cat([q[..., :d] * cos - q[..., d:] * sin, q[..., d:] * cos + q[..., :d] * sin], dim=-1)
        k = torch.cat([k[..., :d] * cos - k[..., d:] * sin, k[..., d:] * cos + k[..., :d] * sin], dim=-1)
        if cache is not None:
            k = torch.cat([cache["kv"][li][0], k], dim=2)
            v = torch.cat([cache["kv"][li][1], v], dim=2)
        new_kv.append((k, v))
        kk, vv = (k.repeat_interleave(a.n_rep, dim=1), v.repeat_interleave(a.n_rep, dim=1)) if a.n_rep > 1 else (k, v)
        scores = (q @ kk.transpose(-2, -1)) * a.scale
        scores = scores.masked_fill(~allowed[:, None], float('-inf'))
        out = (torch.softmax(scores, dim=-1) @ vv).transpose(1, 2).contiguous().view(B, T, -1)
        x = x + a.out_proj(out)
        x = x + blk.ffn(blk.ln2(x))
    return self.head(self.ln_f(x)), {"pos": S, "kv": new_kv, "pad": pad}

def load_model(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
        state = state["model"]                     # full training checkpoint -> weights only
    if MODEL_KIND == "arch2":
        cfg = LMConfig(vocab_size=VOCAB_SIZE, d_model=640, n_layers=12, n_heads=10, n_kv_heads=2, d_ff=1706,
                       max_seq_len=MODEL_SEQ_LEN, dropout=0.0)
    else:
        layers = sorted({int(k.split(".")[1]) for k in state if ".engram." in k})
        cfg = LMConfig(vocab_size=VOCAB_SIZE, engram_layers=tuple(layers))
    m = DecoderLM(cfg)
    m.load_state_dict({k: v.float() for k, v in state.items()}, strict=True)
    if MODEL_KIND == "arch4":
        m.set_engram_compression(ENGRAM_COMPRESS)
    else:
        m.step = types.MethodType(_arch2_step, m)     # cached generation (verified against the full forward below)
        m.supports_pad = True                          # several prompts of different lengths per batch
    return m.to(device).eval()

ENGRAM_COMPRESS = engram_compress_map() if MODEL_KIND == "arch4" else None
CANDIDATES = discover_candidates()
assert CANDIDATES, "no trained checkpoints found: attach the training output(s) of this model family"
if MODEL_KIND == "arch2":
    _m = load_model(next(iter(CANDIDATES.values())))
    _ids = torch.tensor([tokenizer.encode("The appellant contends that the High Court erred in holding that the "
                                          "limitation period under Section 5 of the Limitation Act had expired.")] * 2,
                        device=device)
    with torch.inference_mode():
        _full = _m(_ids).float()
        _lg, _c = _m.step(_ids[:, :-6])
        _steps = [_lg[:, -1]]
        for _t in range(_ids.shape[1] - 6, _ids.shape[1]):
            _lg, _c = _m.step(_ids[:, _t:_t + 1], _c)
            _steps.append(_lg[:, -1])
    _err = (torch.stack(_steps[:-1], 1).float() - _full[:, -7:-1]).abs().max().item()
    assert _err < 1e-3, f"Arch 2 cached generation differs from the full forward ({_err:.2e})"
    print(f"[PASS] Arch 2 KV-cached step == full forward (max logit diff {_err:.1e}, fp32)")
    del _m, _full, _lg, _c, _steps
print("Candidate checkpoints:")
for k, v in CANDIDATES.items():
    print(f"  {k:28s} {v}")


## 9. Generation Engine, Metrics and MBR

In [ ]:
NEVER_GEN = [PAD_ID, UNK_ID, BOS_ID, EOS_ID]


def _filter_row(lg, cfg, generated, ngram_bans):
    """Apply one row's decoding rules to its next-token logits (float32, length V). Returns the filtered logits."""
    lg[NEVER_GEN] = float('-inf')
    rp = cfg["repetition_penalty"]
    if rp != 1.0 and generated:
        ids = torch.tensor(sorted(set(generated[-100:])), device=lg.device)
        vals = lg[ids]
        if cfg.get("rep_mode", "fixed") == "arch2_original":
            lg[ids] = vals / rp                                    # Arch 2's original rule (raises negative logits)
        else:
            lg[ids] = torch.where(vals > 0, vals / rp, vals * rp)  # standard: always lowers repeated tokens
    n = cfg.get("no_repeat_ngram", 0)
    if n and len(generated) >= n - 1:
        banned = ngram_bans.get(tuple(generated[len(generated) - (n - 1):]) if n > 1 else (), ())
        if banned:
            lg[list(banned)] = float('-inf')
    if cfg["temperature"] > 0:
        lg = lg / cfg["temperature"]
        if cfg["top_k"] > 0:
            kth = torch.topk(lg, min(cfg["top_k"], lg.numel())).values[-1]
            lg[lg < kth] = float('-inf')
        if cfg["top_p"] < 1.0:
            sl, si = torch.sort(lg, descending=True)
            cum = torch.cumsum(torch.softmax(sl, dim=-1), dim=-1)
            rm = cum > cfg["top_p"]
            rm[1:] = rm[:-1].clone()
            rm[0] = False
            lg[si[rm]] = float('-inf')
    return lg


MAX_ROWS = 64                      # rows (prompt x decoding x sample) generated together in one batch


def _filter_batch(logits, cfgs, gens, bans, active):
    """The rules of _filter_row applied to all rows at once (same operations, same order):
    never-generate tokens, repetition penalty over the last 100 generated tokens, no-repeat n-gram bans,
    then (sampling rows only) temperature, top-k, top-p. logits: (B, V) float32."""
    B, V = logits.shape
    lg = logits.clone()
    lg[:, NEVER_GEN] = float('-inf')
    dev = lg.device
    rp = torch.tensor([c["repetition_penalty"] for c in cfgs], device=dev, dtype=lg.dtype)[:, None]
    orig = torch.tensor([c.get("rep_mode", "fixed") == "arch2_original" for c in cfgs], device=dev)[:, None]
    pen = torch.zeros(B, V, dtype=torch.bool, device=dev)
    rows_idx, tok_idx = [], []
    for r in range(B):
        if active[r] and cfgs[r]["repetition_penalty"] != 1.0 and gens[r]:
            ids = sorted(set(gens[r][-100:]))
            rows_idx += [r] * len(ids)
            tok_idx += ids
    if rows_idx:
        pen[torch.tensor(rows_idx, device=dev), torch.tensor(tok_idx, device=dev)] = True
        lg = torch.where(pen & orig, lg / rp, lg)                                      # Arch 2's original rule
        lg = torch.where(pen & ~orig, torch.where(lg > 0, lg / rp, lg * rp), lg)       # standard rule
    for r in range(B):
        n = cfgs[r].get("no_repeat_ngram", 0)
        if active[r] and n and len(gens[r]) >= n - 1:
            banned = bans[r].get(tuple(gens[r][len(gens[r]) - (n - 1):]) if n > 1 else (), ())
            if banned:
                lg[r, list(banned)] = float('-inf')
    samp = torch.tensor([c["temperature"] > 0 for c in cfgs], device=dev)
    temp = torch.tensor([c["temperature"] if c["temperature"] > 0 else 1.0 for c in cfgs], device=dev, dtype=lg.dtype)
    lg = torch.where(samp[:, None], lg / temp[:, None], lg)
    ks = [min(c["top_k"], V) if (c["temperature"] > 0 and c["top_k"] > 0) else 0 for c in cfgs]
    ps = [c["top_p"] if c["temperature"] > 0 else 1.0 for c in cfgs]
    rows_k = [r for r in range(B) if ks[r] > 0]
    if rows_k:
        kmax = max(ks[r] for r in rows_k)
        ri = torch.tensor(rows_k, device=dev)
        tv, ti = torch.topk(lg[ri], kmax, dim=-1)                                   # sorted, descending
        kth = tv[torch.arange(len(rows_k), device=dev), torch.tensor([ks[r] - 1 for r in rows_k], device=dev)]
        lg[ri] = lg[ri].masked_fill(lg[ri] < kth[:, None], float('-inf'))
        # top-p over the surviving candidates: after top-k only (a subset of) the top kmax entries are finite,
        # so their sorted order and softmax are those of the full row
        tv = tv.masked_fill(tv < kth[:, None], float('-inf'))
        pk = torch.tensor([ps[r] for r in rows_k], device=dev, dtype=lg.dtype)
        cum = torch.cumsum(torch.softmax(tv, dim=-1), dim=-1)
        rm = cum > pk[:, None]
        rm[:, 1:] = rm[:, :-1].clone()
        rm[:, 0] = False
        rm &= (pk < 1.0)[:, None]
        sub = lg[ri]
        sub.scatter_(-1, ti, sub.gather(-1, ti).masked_fill(rm, float('-inf')))
        lg[ri] = sub
    rows_p = [r for r in range(B) if ks[r] == 0 and ps[r] < 1.0]                    # top-p without top-k: full sort
    if rows_p:
        ri = torch.tensor(rows_p, device=dev)
        sl, si = torch.sort(lg[ri], dim=-1, descending=True)
        cum = torch.cumsum(torch.softmax(sl, dim=-1), dim=-1)
        rm = cum > torch.tensor([ps[r] for r in rows_p], device=dev, dtype=lg.dtype)[:, None]
        rm[:, 1:] = rm[:, :-1].clone()
        rm[:, 0] = False
        lg[ri] = lg[ri].masked_fill(torch.zeros_like(rm).scatter(-1, si, rm), float('-inf'))
    return lg


def _expand(obj, idx):
    """Copy a (nested) cache built for the distinct prompts to the rows: tensors are indexed along dim 0."""
    if torch.is_tensor(obj):
        return obj.index_select(0, idx)
    if isinstance(obj, dict):
        return {k: _expand(v, idx) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return type(obj)(_expand(v, idx) for v in obj)
    return obj


def _generate_batch(model, bases, truncs, cfgs, seeds):
    """One batch of rows; row r continues token list bases[r] with decoding cfgs[r] and seed seeds[r].
    Rows are left-padded to a common length; models with step(..., pad=...) get per-row positions and a padding
    mask, so every row computes exactly what it would compute alone. A row's output depends only on
    (prompt, config, seed): sampling uses per-row Gumbel noise from a per-row generator."""
    B = len(bases)
    L = max(len(b) for b in bases)
    pads = [L - len(b) for b in bases]
    padded = any(pads)
    assert not padded or getattr(model, "supports_pad", False), "this model cannot batch prompts of different lengths"
    gens = [[] for _ in range(B)]
    done = [False] * B
    bans = [dict() for _ in range(B)]
    rngs = [torch.Generator(device=device).manual_seed(int(s)) for s in seeds]
    cached = hasattr(model, "step")
    rows = [[PAD_ID] * p + list(b) for p, b in zip(pads, bases)]
    pad_t = torch.tensor(pads, dtype=torch.long, device=device)
    cache, nxt = None, torch.tensor(rows, dtype=torch.long, device=device)
    with torch.inference_mode():
        for _ in range(min(MAX_NEW_TOKENS, MODEL_SEQ_LEN - L)):
            if all(done):
                break
            with amp_ctx():
                if cached and cache is None:          # prompt pass: once per distinct prompt, then copied to its rows
                    uniq, row_to_u = {}, []
                    for b in bases:
                        row_to_u.append(uniq.setdefault(id(b), len(uniq)))
                    first = [row_to_u.index(u) for u in range(len(uniq))]
                    fi = torch.tensor(first, device=device)
                    if padded:
                        out_, cache = model.step(nxt.index_select(0, fi), None, pad=pad_t.index_select(0, fi))
                    else:
                        out_, cache = model.step(nxt.index_select(0, fi), None)
                    ri = torch.tensor(row_to_u, device=device)
                    cache = _expand(cache, ri)
                    logits = out_[:, -1].float().index_select(0, ri)
                elif cached:
                    out_, cache = model.step(nxt, cache)
                    logits = out_[:, -1].float()
                else:
                    logits = model(torch.tensor(rows, dtype=torch.long, device=device))[:, -1].float()
            active = [not d for d in done]
            lg = _filter_batch(logits, cfgs, gens, bans, active)
            noise = torch.zeros_like(lg)
            for r in range(B):
                if active[r] and cfgs[r]["temperature"] > 0:
                    u = torch.rand(lg.shape[1:], generator=rngs[r], device=lg.device).clamp_(1e-10, 1 - 1e-10)
                    noise[r] = -torch.log(-torch.log(u))       # Gumbel-max = sampling from softmax(lg)
            toks = (lg + noise).argmax(dim=-1).tolist()
            for r in range(B):
                if done[r]:
                    rows[r].append(PAD_ID)          # keep rows equal length; ignored
                    continue
                tok = toks[r]
                rows[r].append(tok)
                gens[r].append(tok)
                n = cfgs[r].get("no_repeat_ngram", 0)
                if n and len(gens[r]) >= n:
                    bans[r].setdefault(tuple(gens[r][-n:-1]), set()).add(tok)
                n_words = len(tokenizer.decode(gens[r]).split())
                if n_words > TARGET_WORDS or (n_words == TARGET_WORDS and tokenizer._ends_word.get(tok, False)):
                    done[r] = True
            nxt = torch.tensor([[row[-1]] for row in rows], dtype=torch.long, device=device)
            del logits, lg, noise
    out = []
    for r in range(B):
        words = tokenizer.decode(gens[r]).split()[:TARGET_WORDS]
        out.append((" ".join(words), {"n_words": len(words), "n_tokens": len(gens[r]),
                                      "hit_token_cap": not done[r], "prompt_truncated": truncs[r]}))
    return out


def generate_many(model, jobs, label=None):
    """jobs: list of (prompt, cfgs, seeds). Returns one [(text, info) per cfg] list per job.
    Jobs are packed into batches of at most MAX_ROWS rows (several prompts per batch when the model supports
    padding, otherwise one prompt per batch)."""
    enc = []
    for prompt, cfgs, seeds in jobs:
        assert len(cfgs) == len(seeds) and 0 < len(cfgs) <= MAX_ROWS
        p_ids = tokenizer.encode(prompt)
        enc.append((p_ids[-PROMPT_KEEP:], len(p_ids) > PROMPT_KEEP))
    multi = getattr(model, "supports_pad", False)
    results, i, t0 = [None] * len(jobs), 0, time.time()
    while i < len(jobs):
        j, rows = i, 0
        while j < len(jobs) and rows + len(jobs[j][1]) <= MAX_ROWS and (multi or j == i):
            rows += len(jobs[j][1])
            j += 1
        bases, truncs, cfgs, seeds, owner = [], [], [], [], []
        for k in range(i, j):
            for c, s in zip(jobs[k][1], jobs[k][2]):
                bases.append(enc[k][0]); truncs.append(enc[k][1]); cfgs.append(c); seeds.append(s); owner.append(k)
        outs = _generate_batch(model, bases, truncs, cfgs, seeds)
        for k in range(i, j):
            results[k] = [o for o, w in zip(outs, owner) if w == k]
        if label and (j // 50 > i // 50 or j == len(jobs)):
            print(f"    [{label}] {j}/{len(jobs)} prompts | {(time.time() - t0)/60:.1f} min | "
                  f"{(time.time() - NOTEBOOK_START)/3600:.2f}h elapsed")
        i = j
    return results


def generate_rows(model, prompt, cfgs, seeds):
    """Generate one continuation per row for the SAME prompt (rows batched together)."""
    return generate_many(model, [(prompt, cfgs, seeds)])[0]


def repetition_rate(text, n=4):
    w = text.split()
    g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not g else 1.0 - len(set(g)) / len(g)


def rouge_f(a, b, n):
    ga = Counter(tuple(a[i:i + n]) for i in range(len(a) - n + 1))
    gb = Counter(tuple(b[i:i + n]) for i in range(len(b) - n + 1))
    ov = sum((ga & gb).values())
    if not ga or not gb or ov == 0:
        return 0.0
    p, r = ov / sum(ga.values()), ov / sum(gb.values())
    return 2 * p * r / (p + r)


def mbr_select(cands):
    """Minimum Bayes Risk choice among a model's own samples: the candidate with the highest average
    ROUGE-1/ROUGE-2 F1 against all other candidates (a consensus pick; no external model involved)."""
    toks = [c.split() for c in cands]
    scores = []
    for i, a in enumerate(toks):
        u = [0.5 * (rouge_f(a, b, 1) + rouge_f(a, b, 2)) for j, b in enumerate(toks) if j != i]
        scores.append(float(np.mean(u)) if u else 0.0)
    return int(np.argmax(scores)), scores


def val_loss(model, max_batches=200):
    """Token-weighted next-token loss on the validation blocks (same batches for every checkpoint)."""
    tot, ntok = 0.0, 0
    with torch.inference_mode():
        # evenly spaced blocks across the whole validation set (blocks are stored source by source)
        idx = np.unique(np.linspace(0, len(val_blocks) - 1, min(len(val_blocks), max_batches * 4)).astype(int))
        for a in range(0, len(idx), 4):
            blk = torch.from_numpy(val_blocks[idx[a:a + 4]].astype(np.int64)).to(device)
            with amp_ctx():
                lg = model(blk[:, :-1])
            tot += F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), blk[:, 1:].reshape(-1), reduction="sum").item()
            ntok += blk[:, 1:].numel()
    return tot / max(1, ntok)


_scorer = None
def bertscore(preds, refs):
    """bert_score package with roberta-large defaults (layer 17, no idf, no baseline rescaling).
    The official evaluator's exact settings are not available here, so absolute values approximate CodaBench;
    every comparison in this notebook uses the same scorer, so rankings are consistent."""
    global _scorer
    if _scorer is None:
        try:
            import bert_score  # noqa: F401
        except ImportError:
            import subprocess
            subprocess.run(['pip', 'install', 'bert_score', '-q'], check=True)
        from bert_score import BERTScorer
        _scorer = BERTScorer(model_type="roberta-large", lang="en", device=device)
    P, R, F1 = _scorer.score(preds, refs)
    return P.numpy(), R.numpy(), F1.numpy()


def summarize(preds, infos, refs, sources=None):
    P, R, F1 = bertscore(preds, refs)
    res = {"F1": float(F1.mean()), "P": float(P.mean()), "R": float(R.mean()),
           "exact_length_rate": float(np.mean([i["n_words"] == TARGET_WORDS for i in infos])),
           "token_cap_hits": int(sum(i["hit_token_cap"] for i in infos)),
           "repetition_4gram": float(np.mean([repetition_rate(p) for p in preds])), "n": len(preds)}
    if sources is not None:
        res["per_source_F1"] = {s: float(np.mean([F1[k] for k in range(len(preds)) if sources[k] == s]))
                                for s in sorted(set(sources))}
    return res, F1


def run_configs(model, examples, cfgs, seed):
    """Generate every config for every example (all configs of a prompt share the prompt's seed; prompts are
    batched together). Returns {config index: (preds, infos)}."""
    out = {c: ([], []) for c in range(len(cfgs))}
    chunks = [cfgs[a:a + MAX_ROWS] for a in range(0, len(cfgs), MAX_ROWS)]
    per_ex = [[] for _ in examples]
    for chunk in chunks:
        jobs = [(ex["prompt"], chunk, [seed * 1_000_003 + i] * len(chunk)) for i, ex in enumerate(examples)]
        for i, res in enumerate(generate_many(model, jobs, label=f"{len(chunk)} configs")):
            per_ex[i].extend(res)
    for i in range(len(examples)):
        for c, (t, info) in enumerate(per_ex[i]):
            out[c][0].append(t)
            out[c][1].append(info)
    return out

print("Generation engine ready (batched rows, per-row seeds, special tokens never generated, exact-word stopping).")


## Model and references

In [ ]:
# ---- the model and the held-out references (no checkpoint search: the pool uses the original Arch 2) ----
SELECTED_CKPT = "arch2_continued" if USE_CONTINUED and "arch2_continued" in CANDIDATES else "arch2_original"
assert SELECTED_CKPT in CANDIDATES, f"attach the Arch 2 output; found {list(CANDIDATES)}"
refs = [e["reference"] for e in eval_set]
sources = [e["source"] for e in eval_set]
print(f"Model: {SELECTED_CKPT} ({CANDIDATES[SELECTED_CKPT]}) | held-out tuning prompts: {len(eval_set)}")


## Candidate pool, 18 selectors, 18 submissions

In [ ]:
# ---- One candidate pool per prompt, 18 selectors, one validated submission per selector ----
from huggingface_hub import hf_hub_download

CFG = {
    "t35": {"temperature": 0.5, "top_p": 0.95, "top_k": 20, "repetition_penalty": 1.20, "no_repeat_ngram": 0, "rep_mode": "fixed"},
    "t36": {"temperature": 0.3, "top_p": 0.85, "top_k": 50, "repetition_penalty": 1.20, "no_repeat_ngram": 0, "rep_mode": "fixed"},
    "submitted": {"temperature": 0.4, "top_p": 0.92, "top_k": 50, "repetition_penalty": 1.15, "no_repeat_ngram": 0, "rep_mode": "arch2_original"},
    "t34": {"temperature": 0.6, "top_p": 0.95, "top_k": 20, "repetition_penalty": 1.10, "no_repeat_ngram": 0, "rep_mode": "fixed"},
}
POOL = [("t35", 48)]                                               # 48 candidates per prompt, all with t35
ROW_CFG = [c for c, n in POOL for _ in range(n)]
C = list(range(len(ROW_CFG)))
SELECTORS = {                                  # name -> candidate indices (MBR with ROUGE-1/2 consensus)
    "mbr48": C,
    "mbr40a": C[:40],          "mbr40b": C[8:],
    "mbr32a": C[:32],          "mbr32b": C[16:],          "mbr32c": C[:16] + C[32:],
    "mbr24a": C[:24],          "mbr24b": C[24:],          "mbr24c": C[12:36],
    "mbr20a": C[:20],          "mbr20b": C[20:40],
    "mbr16a": C[:16],          "mbr16b": C[16:32],        "mbr16c": C[32:],
    "mbr12a": C[:12],          "mbr12b": C[12:24],        "mbr12c": C[24:36],        "mbr12d": C[36:],
}
assert len(SELECTORS) == 18
CHUNK = 50                                     # prompts generated together; progress is reported per chunk
MAX_ROWS = 96                                  # 2 prompts x 48 candidates per generation batch

def pairwise(cands):
    """ROUGE-1/2 utility for every candidate pair of one prompt (the decoding notebook's MBR utility)."""
    toks = [c.split() for c in cands]
    n = len(cands)
    M = np.zeros((n, n))
    for a in range(n):
        for b in range(a + 1, n):
            M[a, b] = M[b, a] = 0.5 * (rouge_f(toks[a], toks[b], 1) + rouge_f(toks[a], toks[b], 2))
    return M

def select_all(outs):
    """outs: 28 (text, info) candidates of one prompt (+ its utility matrices) -> {selector: (text, info)}."""
    cands, R = outs
    res = {}
    for name, idx in SELECTORS.items():
        if len(idx) == 1:
            res[name] = cands[idx[0]]
            continue
        ok = [i for i in idx if cands[i][1]["n_words"] == TARGET_WORDS] or idx
        score = [np.mean([R[i, j] for j in ok if j != i]) if len(ok) > 1 else 0.0 for i in ok]
        res[name] = cands[ok[int(np.argmax(score))]]
    return res

def pool_select(model, prompts, seed_base, label):
    """Generate the 28-candidate pool for every prompt (batched), then apply all selectors. Returns one
    {selector: (text, info)} dict per prompt."""
    results = []
    t0 = time.time()
    for a in range(0, len(prompts), CHUNK):
        ps = prompts[a:a + CHUNK]
        jobs = [(p, [CFG[c] for c in ROW_CFG], [seed_base + (a + i) * 100 + k for k in range(len(ROW_CFG))])
                for i, p in enumerate(ps)]
        outs = generate_many(model, jobs)
        for o in outs:
            results.append(select_all((o, pairwise([t for t, _ in o]))))
        print(f"    [{label}] {min(a + CHUNK, len(prompts))}/{len(prompts)} prompts | {(time.time() - t0)/60:.1f} min | "
              f"{(time.time() - NOTEBOOK_START)/3600:.2f}h elapsed")
    return results

model = load_model(CANDIDATES[SELECTED_CKPT])

# ---- 1. held-out tuning prompts: score every selector (ranking for the submission order) ----
held_path = os.path.join(OUTPUT_DIR, "pool_heldout.json")
if os.path.exists(held_path):
    ranking = json.load(open(held_path))
else:
    sel = pool_select(model, [e["prompt"] for e in eval_set], 18_000_000, "held-out")
    ranking = {}
    for name in SELECTORS:
        r, _ = summarize([s[name][0] for s in sel], [s[name][1] for s in sel], refs, sources)
        ranking[name] = r
    save_json(ranking, "pool_heldout.json")
order = sorted(ranking, key=lambda k: -ranking[k]["F1"])
print(f"\n{'rank':>4s} {'held-out F1':>11s} {'exact128':>8s} {'rep4':>6s}  selector")
for k, name in enumerate(order, 1):
    r = ranking[name]
    print(f"{k:4d} {r['F1']:11.4f} {r['exact_length_rate']:8.2f} {r['repetition_4gram']:6.3f}  {name}")

# ---- 2. official test prompts: one submission zip per selector (saved after every chunk) ----
test_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
test_ids = [str(k) for k in test_data.keys()]
assert len(test_ids) == len(set(test_ids)), "duplicate test IDs in the official file"
part_path = os.path.join(OUTPUT_DIR, "pool_test_partial.json")
part = json.load(open(part_path)) if os.path.exists(part_path) else {n: {} for n in SELECTORS}
todo = [tid for tid in test_ids if not all(tid in part[n] for n in SELECTORS)]
t0 = time.time()
for a in range(0, len(todo), 200):
    ids = todo[a:a + 200]
    sel = pool_select(model, [test_data[t]["text"] for t in ids], 19_000_000 + test_ids.index(ids[0]) * 100, "test")
    for tid, s in zip(ids, sel):
        for name in SELECTORS:
            part[name][tid] = {"text": s[name][0]}
    save_json(part, "pool_test_partial.json")

# any submission.zip attached as input (e.g. the earlier MBR submission): report overlap so no slot is wasted on a duplicate
prev_zips = sorted(glob.glob(f"{INPUT_ROOT}/**/*.zip", recursive=True))
prevs = {}
for p in prev_zips:
    try:
        with zipfile.ZipFile(p) as z:
            if z.namelist() == ["prediction.json"]:
                prevs[p] = json.loads(z.read("prediction.json"))
    except (zipfile.BadZipFile, KeyError, json.JSONDecodeError):
        pass
for k, name in enumerate(order, 1):
    preds = {tid: part[name][tid] for tid in test_ids}
    assert len(preds) == len(test_ids) and all(isinstance(p["text"], str) and p["text"] for p in preds.values())
    assert not any(s in p["text"] for p in preds.values() for s in ("<s>", "</s>", "<pad>", "<unk>"))
    zpath = os.path.join(OUTPUT_DIR, f"{k:02d}_submission_{name}.zip")      # file name starts with its held-out rank
    pj = os.path.join(OUTPUT_DIR, "prediction.json")
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"] and json.loads(z.read("prediction.json")) == preds
    os.remove(pj)
    lens = Counter(len(p["text"].split()) for p in preds.values())
    if lens != Counter({TARGET_WORDS: len(preds)}):      # still a valid submission: report it, never abort the remaining zips
        print(f"  WARNING {name}: not all predictions are exactly {TARGET_WORDS} words: {dict(sorted(lens.items()))}")
    overlap = {os.path.relpath(p, INPUT_ROOT): sum(pv.get(t, {}).get("text") == preds[t]["text"] for t in test_ids) for p, pv in prevs.items()}
    print(f"READY {os.path.basename(zpath)} | held-out F1 {ranking[name]['F1']:.4f} | exactly 128 words "
          f"{lens.get(TARGET_WORDS, 0)}/{len(preds)}" + (f" | predictions identical to attached zips: {overlap}" if overlap else ""))
distinct = len({json.dumps([part[n][t]["text"] for t in test_ids]) for n in SELECTORS})
if distinct < len(SELECTORS):                            # report duplicates (zips are already written; skip the later copy)
    seen = {}
    for k, name in enumerate(order, 1):
        key = json.dumps([part[name][t]["text"] for t in test_ids])
        if key in seen:
            print(f"  DUPLICATE: {k:02d}_submission_{name}.zip has the same predictions as {seen[key]} -> do not submit it")
        else:
            seen[key] = f"{k:02d}_submission_{name}.zip"
print(f"\n{len(SELECTORS)} submissions written to {OUTPUT_DIR}; {distinct} have distinct predictions. "
      f"Submit in file-name order (best held-out F1 first).")
